### Middleware

Middleware provides a way to more tightly control what happens inside the agent. Middleware is useful for the following:

- Tracking agent behaviour with logging, analytics and debugging.
- Transforming prompts, tool sections and output formatting.
- Adding retries, fallbacks, and early terminating logic.
- Applying rate limitings, guardrails and PII data protection.

In [36]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
os.environ["GEMINI_API_KEY"] = os.getenv("GEMINI_API_KEY")

model = init_chat_model("groq:qwen/qwen3-32b")


## Summarization Middleware

Built-in middleware in langchain which creates a summarization of the conversation when approaching the token limit.

It is useful in the following cases:
- long-running conversations that exceeds the context windows.
- multi-turn dialogues with extensive history.
- Application where preserving full conversation context matters.

In [26]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import SystemMessage, HumanMessage


agent = create_agent(
    model=model,
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=model,
            trigger=("messages",6), # triggers after this conversation messages hit in the messages array.
            keep=("messages",4) # keeps last 4 messages in the context for summarization.
        )
    ]
)

Here Trigger happens when in the messages array 10 conversation between System, Human and AI happens
Here keep is for keeping the last n number of conversation from the messages array for summarization.

In [27]:
### Run with thread id
config = {"configurable":{"thread_id":"test_1"}}

In [28]:
# Alternative test data
questions = [
    "What is 2+2?",
    "What is 5*10",
    "What is 10-3",
    "What is 64/4",
    "What is the color of sky",
]

In [29]:
for q in questions:
    response = agent.invoke({"messages":[HumanMessage(content=q)]},config)
    print(f"messages : {response}")
    print(f"messages len: {len(response["messages"])}")

messages : {'messages': [HumanMessage(content='What is 2+2?', additional_kwargs={}, response_metadata={}, id='4e1ee893-b9f5-4214-84a5-8fbd7eb773ef'), AIMessage(content='2 + 2 = 4', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019f4801-b520-7a11-b088-65eff46f376c-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 8, 'output_tokens': 27, 'total_tokens': 35, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 20}})]}
messages len: 2
messages : {'messages': [HumanMessage(content='What is 2+2?', additional_kwargs={}, response_metadata={}, id='4e1ee893-b9f5-4214-84a5-8fbd7eb773ef'), AIMessage(content='2 + 2 = 4', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019f4801-b520-7a11-b088-65eff46f376c-0', 


KeyboardInterrupt



## Now summarization middleware based on token size

In [37]:
from langchain.agents import create_agent
from langchain.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage
from langchain.agents.middleware import SummarizationMiddleware
from langchain.chat_models import init_chat_model

@tool
def search_hotels(city: str) -> str:
    """Search Hotels - return long response to use more tokens"""
    return f"""Hotels in city: {city}:
    1. Grand Hotels - 5 Stars, $350/night, spa, pool, wifi, gym.
    2. City Inn - 3 stars, $200/night, pool, wifi.
    3. Budget Stay - 2 Stars, $50/night, wifi, cafe.
    """
    

llm = init_chat_model("groq:qwen/qwen3-32b")

agent = create_agent(
    model=llm,
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=llm,
            trigger=("tokens",550),
            keep=("tokens",200)
        )
    ]
)

config = {"configurable":{"thread_id":"test-1"}} # keeps the conversation for same user

def count_tokens(messages:str):
    total_char = sum(len(str(m.content)) for m in messages)
    return total_char // 4  # 4 char = 1 token

In [39]:
# Run test
cities = ["Paris","London","New York","Dubai"]


for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"find hotels in {city}")]},
        config
    )
    
    tokens = count_tokens(response["messages"])
    print(f"{city}: ~{tokens}, {len(response["messages"])} messages")
    print(f"{response["messages"]}")

KeyboardInterrupt: 

## Fraction
Now summarization takes place based on fraction of context window

In [ ]:
from langchain.agents import create_agent
from langchain.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage
from langchain.agents.middleware import SummarizationMiddleware
from langchain.chat_models import init_chat_model

@tool
def search_hotels(city: str) -> str:
    """Search Hotels - return long response to use more tokens"""
    return f"""Hotels in city: {city}:
    1. Grand Hotels - 5 Stars, $350/night, spa, pool, wifi, gym.
    2. City Inn - 3 stars, $200/night, pool, wifi.
    3. Budget Stay - 2 Stars, $50/night, wifi, cafe.
    """
    

llm = init_chat_model("groq:qwen/qwen3-32b")

agent = create_agent(
    model=llm,
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=llm,
            trigger=("fraction",0.005),  # 0.005% of the context window for the model
            keep=("fraction",0.002) # 0.002@ of the context window for the model
        )
    ]
)

config = {"configurable":{"thread_id":"test-1"}} # keeps the conversation for same user

def count_tokens(messages:str):
    total_char = sum(len(str(m.content)) for m in messages)
    return total_char // 4  # 4 char = 1 token


# Run test
cities = ["Paris","London","New York","Dubai"]


for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"find hotels in {city}")]},
        config
    )
    
    tokens = count_tokens(response["messages"])
    fractions = tokens / 128000  # context size of gpt-4 model
    print(f"{city}: ~{tokens}, {len(response["messages"])} messages")
    print(f"{response["messages"]}")